# hook函数执行顺序

## 顺序规则
- agent的before始终优先于model的中间件执行，after始终晚于model中间件执行
- 触发工具调用时，不会再执行agent的中间件，tool的中间件优先于model中间件执行
- 同一类型（agent\tool\model，且无论是None-style还是Wrap-style）的中间件，before按照声明顺序执行，after按照声明逆序执行

In [8]:
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain_core.tools import tool
from langchain.agents.middleware import (
    before_model,
    after_model,
    AgentState,
    wrap_model_call,
    ModelRequest,
    ModelResponse, before_agent, after_agent, wrap_tool_call,
)
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any, Callable

from common import init_simple_dashscope_model

@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res

@before_model
def before_model_middleware3(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model-3 <- "
    return None


@before_model
def before_model_middleware1(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model-1 <- "
    return None


@before_model
def before_model_middleware2(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model-2 <- "
    return None

@before_agent
def before_agent_middleware1(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_agent_middleware1 <- "
    return None

@before_agent
def before_agent_middleware2(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_agent_middleware2 <- "
    return None

@after_agent
def after_agent_middleware1(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_agent_middleware1 <- "
    return None

@after_agent
def after_agent_middleware2(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_agent_middleware2 <- "
    return None

@after_model
def after_model_middleware2(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model-2 <- "
    return None


@after_model
def after_model_middleware1(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model-1 <- "
    return None


@after_model
def after_model_middleware3(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model-3 <- "
    return None


@wrap_model_call
def wrap_model_middleware1(request: ModelRequest,
                           handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse | None:
    request.messages[-1].content += " -> wrap_model-before-1 <- "
    response = handler(request)
    response.result[0].content += " -> wrap_model-after-1 <- "
    return response


@wrap_model_call
def wrap_model_middleware3(request: ModelRequest,
                           handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse | None:
    request.messages[-1].content += " -> wrap_model-before-3 <- "
    response = handler(request)
    response.result[0].content += " -> wrap_model-after-3 <- "
    return response


@wrap_model_call
def wrap_model_middleware2(request: ModelRequest,
                           handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse | None:
    request.messages[-1].content += " -> wrap_model-before-2 <- "
    response = handler(request)
    response.result[0].content += " -> wrap_model-after-2 <- "
    return response

@wrap_tool_call
def wrap_tool_call1(request: ToolCallRequest,
                          handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                          ) -> ToolMessage | Command[Any]:
    response = handler(request)
    if isinstance(response, ToolMessage) and isinstance(response.content, str):
        # 进、出各打一个标记；最终会嵌套在 ToolMessage 里
        response.content = (
            f"-> wrap_tool_call1 <- {response.content} -> wrap_tool_call1 <-"
        )
    return response
@wrap_tool_call
def wrap_tool_call2(request: ToolCallRequest,
                          handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                          ) -> ToolMessage | Command[Any]:
    response = handler(request)
    if isinstance(response, ToolMessage) and isinstance(response.content, str):
        response.content = (
            f"-> wrap_tool_call2 <- {response.content} -> wrap_tool_call2 <-"
        )
    return response


model = init_simple_dashscope_model('qwen-max')

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[

        before_model_middleware1,
        before_model_middleware2,
        after_model_middleware2,
        after_model_middleware1,
        wrap_model_middleware2,
        wrap_model_middleware1,
        before_agent_middleware1,
        before_agent_middleware2,
        after_agent_middleware2,
        after_agent_middleware1,
        wrap_tool_call2,
        wrap_tool_call1,
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("今天深圳天气怎么样？")],
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天深圳天气怎么样？ -> before_agent_middleware1 <-  -> before_agent_middleware2 <-  -> before_model-1 <-  -> before_model-2 <-  -> wrap_model-before-2 <-  -> wrap_model-before-1 <- 
================================== Ai Message ==================================

 -> wrap_model-after-1 <-  -> wrap_model-after-2 <-  -> after_model-1 <-  -> after_model-2 <-
Tool Calls:
  get_weather (call_874ce6314f004a6d886677)
 Call ID: call_874ce6314f004a6d886677
  Args:
    city: 深圳
    is_forcast: False
================================= Tool Message =================================
Name: get_weather

-> wrap_tool_call2 <- -> wrap_tool_call1 <- 深圳今天天气不错 -> wrap_tool_call1 <- -> wrap_tool_call2 <- -> before_model-1 <-  -> before_model-2 <-  -> wrap_model-before-2 <-  -> wrap_model-before-1 <- 
================================== Ai Message ==================================

深圳今天天气不错。 -> wrap_model-after-1 <-  -> wrap_model-after